# SkinCon concept bottleneck models, hyperparameter tuning with a DDI ablation

This notebook searches for the strongest concept bottleneck model under a fixed GPU budget, then exports three seeds in exactly the format that `AIA_SkinCon.ipynb` reads. It reuses the frozen split of the analysis notebook, so every result stays comparable.

**Stage A, search (seed 42).** Five configurations, a base setting plus a one-factor-at-a-time design around it. Every configuration first trains 12 of 40 epochs of its cosine schedule. The three best on validation continue from their checkpoint to 40 epochs (successive halving), so no epoch is wasted.

**Stage B, DDI ablation.** The best configuration is retrained on Fitzpatrick17k training images only. Both arms are compared on the same validation images, then reported on the test images of each source.

**Stage C, final models.** Head options are chosen on validation (vocabulary of 12, 16 or 20 concepts, inputs from predicted or annotated concepts, per-concept thresholds), then three seeds of the final configuration are trained and exported with the five diagnosis heads of the analysis.

The ranking criterion is the validation AUROC of the published two-layer head on 12 predicted concepts, which is the model the analysis explains. Test images are never used for a decision. Every trial is checkpointed on Drive, so a disconnected session resumes where it stopped.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
# ============================================================
# GLOBAL SETTINGS
# ============================================================
SEEDS = [42, 64, 128]
BACKBONE_NAME = "efficientnet_b3"
USE_DDI = True                 # keep True, the DDI ablation of Stage B decides whether DDI images join training
DOWNLOAD_FITZ_IMAGES = False
TIME_BUDGET_HOURS = 14.0       # the notebook skips optional trials when the projected time exceeds this budget

PROJECT_DIR = "/content/drive/MyDrive/Notebooks/SkinCon"
SKINCON_DIR = f"{PROJECT_DIR}/skincon"
FITZ_META = f"{PROJECT_DIR}/fitzpatrick17k/fitzpatrick17k.csv"
FITZ_IMAGES = f"{PROJECT_DIR}/fitzpatrick17k/images"
DDI_DIR = f"{PROJECT_DIR}/ddi"
MODELS_PATH = f"{PROJECT_DIR}/models"             # tuned models, the analysis notebook reads this folder
TUNING_PATH = f"{PROJECT_DIR}/tuning"             # checkpoints, trial logs plus the summary archive

## Data

Identical to the analysis notebook. The frozen split file on Drive is reused, and the images are copied once to the local disk.

In [3]:
# ============================================================
# DATA. SKINCON ANNOTATIONS, IMAGES, DUPLICATE GROUPS, SPLITS AND CONCEPT VOCABULARIES
# The split file is written once and reused by every later run, so every seed, head and experiment sees the
# same training, validation and test images. Delete SPLIT_FILE to rebuild it.
# ============================================================
import os
import io
import sys
import json
import time
import math
import random
import hashlib
import warnings
import builtins

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")
try:
    import imagehash
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)
    import imagehash
from sklearn.model_selection import StratifiedGroupKFold

original_print = builtins.print
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

SPLIT_SEED = 2024                 # fixed, independent of the model seeds
PHASH_MAX_BITS = 6                # perceptual hashes within this Hamming distance join one duplicate group
MIN_CONCEPT_IMAGES = 50           # concepts with fewer positive images in SkinCon are never used
VOCAB_SIZES = (12, 16, 20)        # nested vocabularies of the most frequent training concepts
SPLIT_DIR = os.path.join(PROJECT_DIR, "splits")
SPLIT_FILE = os.path.join(SPLIT_DIR, f"skincon_splits_{'fitz_ddi' if USE_DDI else 'fitz'}.csv")
VOCAB_FILE = SPLIT_FILE.replace(".csv", "_vocabularies.json")
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

LOCAL_IMAGE_DIR = "/content/skincon_images"   # fast local copy of every image (Google Drive is slow per file)
IO_THREADS = 32                                # parallel reads from Drive and parallel hashing

EXCLUDE_COLUMN = "Do not consider this image"
ANNOT_SKIP = {"Unnamed: 0", "ImageID", EXCLUDE_COLUMN}


def concept_key(name):
    """Column-safe concept key, e.g. 'Brown(Hyperpigmentation)' -> 'brown_hyperpigmentation'."""
    k = "".join(ch.lower() if ch.isalnum() else "_" for ch in name)
    while "__" in k:
        k = k.replace("__", "_")
    return k.strip("_")


def download_fitz_images(df, target_dir, timeout=20):
    """Best-effort download of missing Fitzpatrick17k images from the metadata URLs (many URLs no longer work)."""
    import requests
    os.makedirs(target_dir, exist_ok=True)
    ok = fail = 0
    for _, r in df.iterrows():
        dst = os.path.join(target_dir, r["ImageID"])
        if os.path.exists(dst):
            continue
        try:
            resp = requests.get(r["url"], timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            Image.open(io.BytesIO(resp.content)).convert("RGB").save(dst, quality=95)
            ok += 1
        except Exception:
            fail += 1
    original_print(f"Fitzpatrick17k download: {ok} new images, {fail} failures")


def stage_images(files_by_source):
    """
    Copy every image once from Google Drive to the local disk, with parallel threads and progress output.
    Reading thousands of small files from Drive one at a time takes hours, local reads take milliseconds.
    Files already present locally are skipped, so a restarted session only copies what is missing.
    A zip archive <folder>.zip next to an image folder is extracted instead, which is faster still.
    """
    import shutil
    import zipfile
    from concurrent.futures import ThreadPoolExecutor
    sources = {"fitzpatrick17k": FITZ_IMAGES, "ddi": DDI_DIR}
    found = {}
    for src, files in files_by_source.items():
        dst_dir = os.path.join(LOCAL_IMAGE_DIR, src)
        os.makedirs(dst_dir, exist_ok=True)
        zpath = sources[src].rstrip("/") + ".zip"
        todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo and os.path.exists(zpath):
            t0 = time.time()
            with zipfile.ZipFile(zpath) as zf:
                names = {os.path.basename(n): n for n in zf.namelist() if not n.endswith("/")}
                for f in todo:
                    if f in names:
                        with zf.open(names[f]) as a, open(os.path.join(dst_dir, f), "wb") as b:
                            shutil.copyfileobj(a, b)
            original_print(f"  {src}: extracted from {zpath} in {time.time() - t0:.0f} s")
            todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo:
            try:
                on_drive = set(os.listdir(sources[src]))           # one directory listing instead of one call per file
            except FileNotFoundError:
                on_drive = set()
            copy = [f for f in todo if f in on_drive]
            t0, done = time.time(), [0]

            def one(f):
                shutil.copyfile(os.path.join(sources[src], f), os.path.join(dst_dir, f))
                done[0] += 1
                if done[0] % 250 == 0 or done[0] == len(copy):
                    el = time.time() - t0
                    original_print(f"  {src}: {done[0]}/{len(copy)} images copied, {el:.0f} s elapsed, "
                                   f"about {el / done[0] * (len(copy) - done[0]):.0f} s left", flush=True)

            original_print(f"  {src}: copying {len(copy)} images from Drive with {IO_THREADS} threads", flush=True)
            with ThreadPoolExecutor(IO_THREADS) as ex:
                list(ex.map(one, copy))
        local = set(os.listdir(dst_dir))
        found[src] = {f for f in files if f in local}
        original_print(f"  {src}: {len(found[src])} of {len(files)} images available locally", flush=True)
    return found


def load_skincon():
    """One row per usable SkinCon image with the binary target, the 48 concepts, the source and the image path."""
    fa = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_fitzpatrick17k.csv"))
    concepts = [c for c in fa.columns if c not in ANNOT_SKIP]
    meta = pd.read_csv(FITZ_META)
    fa["md5hash"] = fa["ImageID"].astype(str).str.replace(".jpg", "", regex=False)
    f = fa.merge(meta[["md5hash", "three_partition_label", "label", "fitzpatrick_scale", "qc", "url"]],
                 on="md5hash", how="left", validate="one_to_one")
    counts = {"fitzpatrick17k annotated": len(f)}
    f = f[f[EXCLUDE_COLUMN].fillna(0) == 0]
    counts["fitzpatrick17k after exclusion flag"] = len(f)
    f = f[~f["qc"].astype(str).str.startswith("3")]            # quality control: wrongly labelled
    counts["fitzpatrick17k after quality control"] = len(f)
    f = f[f["three_partition_label"].notna()]
    if DOWNLOAD_FITZ_IMAGES:
        download_fitz_images(f, FITZ_IMAGES)
    f = pd.DataFrame({"image_id": "fitz_" + f["md5hash"], "source": "fitzpatrick17k",
                      "file": f["ImageID"].astype(str).values,
                      "malignant": (f["three_partition_label"] == "malignant").astype(int).values,
                      "diagnosis": f["label"].values, "skin_type": f["fitzpatrick_scale"].astype(str).values,
                      **{concept_key(c): f[c].fillna(0).astype(int).values for c in concepts}})
    frames = [f]
    if USE_DDI:
        da = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_ddi.csv"))
        dm = pd.read_csv(os.path.join(DDI_DIR, "ddi_metadata.csv"))
        d = da.merge(dm, left_on="ImageID", right_on="DDI_file", how="left", validate="one_to_one")
        counts["ddi annotated"] = len(d)
        d = d[(d[EXCLUDE_COLUMN].fillna(0) == 0) & d["malignant"].notna()]
        counts["ddi after exclusion flag"] = len(d)
        mal = d["malignant"].astype(str).str.lower().isin(["true", "1", "1.0", "yes"]).astype(int)
        d = pd.DataFrame({"image_id": "ddi_" + d["ImageID"].astype(str), "source": "ddi",
                          "file": d["ImageID"].astype(str).values,
                          "malignant": mal.values, "diagnosis": d["disease"].values,
                          "skin_type": d["skin_tone"].astype(str).values,
                          **{concept_key(c): d[c].fillna(0).astype(int).values for c in concepts}})
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    original_print("Staging images on the local disk", flush=True)
    found = stage_images({s: df.loc[df["source"] == s, "file"].tolist() for s in df["source"].unique()})
    df["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(df["source"], df["file"])]
    exists = pd.Series([f in found[s] for s, f in zip(df["source"], df["file"])], index=df.index)
    counts["images found on disk"] = int(exists.sum())
    if (~exists).any():
        original_print(f"Missing image files: {int((~exists).sum())} (listed in missing_images.csv, dropped)")
        df[~exists][["image_id", "source", "file"]].to_csv(os.path.join(SPLIT_DIR, "missing_images.csv"), index=False)
    df = df[exists].reset_index(drop=True)
    return df, [concept_key(c) for c in concepts], dict(zip([concept_key(c) for c in concepts], concepts)), counts


def duplicate_groups(paths, max_bits):
    """Union-find over identical files and perceptual hashes within max_bits Hamming distance."""
    from concurrent.futures import ThreadPoolExecutor
    n = len(paths)
    t0 = time.time()

    def one(p):
        with open(p, "rb") as fh:                 # a single read gives both the byte hash and the image
            raw = fh.read()
        with Image.open(io.BytesIO(raw)) as im:
            h = int(str(imagehash.phash(im.convert("RGB"))), 16)
        return h, hashlib.md5(raw).hexdigest()

    out = []
    with ThreadPoolExecutor(IO_THREADS) as ex:
        for k, r in enumerate(ex.map(one, paths), 1):
            out.append(r)
            if k % 500 == 0 or k == n:
                original_print(f"  hashed {k}/{n} images, {time.time() - t0:.0f} s", flush=True)
    ph = np.array([h for h, _ in out], dtype=np.uint64)
    md5 = [m for _, m in out]
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)

    first = {}
    for i, h in enumerate(md5):
        if h in first:
            union(i, first[h])
        else:
            first[h] = i
    bytes_ = ph.view(np.uint8).reshape(n, 8)
    pairs = 0
    for i in range(0, n, 256):
        x = np.bitwise_xor(bytes_[i:i + 256, None, :], bytes_[None, :, :])        # (b, n, 8)
        dist = np.unpackbits(x, axis=-1).sum(-1)
        for a, b in zip(*np.where(dist <= max_bits)):
            a = a + i
            if a < b:
                union(a, b)
                pairs += 1
    return np.array([find(i) for i in range(n)]), pairs


def build_splits():
    df, all_concepts, concept_names, counts = load_skincon()
    t0 = time.time()
    groups, pairs = duplicate_groups(df["path"].tolist(), PHASH_MAX_BITS)
    df["group"] = groups
    sizes = df.groupby("group").size()
    counts["near-duplicate pairs"] = pairs
    counts["duplicate groups with more than one image"] = int((sizes > 1).sum())
    counts["images in such groups"] = int(sizes[sizes > 1].sum())
    mixed = df.groupby("group")["malignant"].nunique()
    counts["groups with conflicting targets"] = int((mixed > 1).sum())
    original_print(f"Perceptual hashing of {len(df)} images: {time.time() - t0:.0f} s")
    strat = df["malignant"].astype(str) + "_" + df["source"]
    sgk = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
    folds = np.zeros(len(df), dtype=int)
    for k, (_, idx) in enumerate(sgk.split(df, strat, df["group"])):
        folds[idx] = k
    df["split"] = np.where(folds == 0, "test", np.where(folds == 1, "valid", "train"))
    # concept vocabularies from the training split only (ties broken by the SkinCon total, then by name)
    tr = df[df["split"] == "train"]
    eligible = [c for c in all_concepts if df[c].sum() >= MIN_CONCEPT_IMAGES]
    order = sorted(eligible, key=lambda c: (-int(tr[c].sum()), -int(df[c].sum()), c))
    if len(order) < max(VOCAB_SIZES):
        raise ValueError(f"Only {len(order)} concepts have at least {MIN_CONCEPT_IMAGES} images.")
    vocab = {f"V{k}": order[:k] for k in VOCAB_SIZES}
    info = {"vocabularies": vocab, "concept_names": {c: concept_names[c] for c in order},
            "training_counts": {c: int(tr[c].sum()) for c in order}, "eligible": eligible,
            "counts": counts, "split_seed": SPLIT_SEED, "phash_max_bits": PHASH_MAX_BITS, "use_ddi": USE_DDI}
    keep = ["image_id", "source", "file", "malignant", "diagnosis", "skin_type", "group", "split"] + all_concepts
    df[keep].to_csv(SPLIT_FILE, index=False)
    with open(VOCAB_FILE, "w") as fh:
        json.dump(info, fh, indent=1)
    return df[keep + ["path"]], info


if os.path.exists(SPLIT_FILE) and os.path.exists(VOCAB_FILE):
    DATA = pd.read_csv(SPLIT_FILE)
    with open(VOCAB_FILE) as fh:
        VOCAB_INFO = json.load(fh)
    original_print(f"Reusing the frozen split file {SPLIT_FILE}")
    original_print("Staging images on the local disk (a new Colab session starts with an empty disk)", flush=True)
    _found = stage_images({s: DATA.loc[DATA["source"] == s, "file"].tolist() for s in DATA["source"].unique()})
    _miss = sum(f not in _found[s] for s, f in zip(DATA["source"], DATA["file"]))
    if _miss:
        raise FileNotFoundError(f"{_miss} images of the frozen split are missing on Drive.")
    DATA["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(DATA["source"], DATA["file"])]
else:
    DATA, VOCAB_INFO = build_splits()

VOCABS = VOCAB_INFO["vocabularies"]
CONCEPTS = VOCABS[f"V{max(VOCAB_SIZES)}"]           # concepts predicted by the concept model (largest vocabulary)
CONCEPT_NAMES = VOCAB_INFO["concept_names"]

original_print("\nData assembly")
for k, v in VOCAB_INFO["counts"].items():
    original_print(f"  {k:<45} {v}")
tab = DATA.groupby(["split", "source", "malignant"]).size().unstack(fill_value=0)
original_print("\nImages per split, source and target (1 = malignant)\n" + tab.to_string())
original_print("\nConcept vocabularies (nested, ordered by training frequency)")
for c in CONCEPTS:
    member = [k for k in VOCABS if c in VOCABS[k]]
    original_print(f"  {CONCEPT_NAMES[c]:<28} train={VOCAB_INFO['training_counts'][c]:>5}   "
                   f"prevalence={DATA[c].mean():.3f}   in {', '.join(member)}")

Reusing the frozen split file /content/drive/MyDrive/Notebooks/SkinCon/splits/skincon_splits_fitz_ddi.csv
Staging images on the local disk (a new Colab session starts with an empty disk)
  fitzpatrick17k: copying 3227 images from Drive with 32 threads
  fitzpatrick17k: 250/3227 images copied, 50 s elapsed, about 599 s left
  fitzpatrick17k: 500/3227 images copied, 270 s elapsed, about 1473 s left
  fitzpatrick17k: 750/3227 images copied, 358 s elapsed, about 1183 s left
  fitzpatrick17k: 1000/3227 images copied, 359 s elapsed, about 799 s left
  fitzpatrick17k: 1250/3227 images copied, 359 s elapsed, about 568 s left
  fitzpatrick17k: 1500/3227 images copied, 360 s elapsed, about 414 s left
  fitzpatrick17k: 1750/3227 images copied, 360 s elapsed, about 304 s left
  fitzpatrick17k: 2000/3227 images copied, 360 s elapsed, about 221 s left
  fitzpatrick17k: 2250/3227 images copied, 361 s elapsed, about 157 s left
  fitzpatrick17k: 2500/3227 images copied, 361 s elapsed, about 105 s left


## Resized image cache

Training reads every image about 25 times per trial. Decoding full-resolution photographs would make the CPU the bottleneck, so every image is resized once to a short side of 400 pixels (enough for the largest input size of the search) and stored locally.

In [4]:
# ============================================================
# RESIZED IMAGE CACHE
# ============================================================
from concurrent.futures import ThreadPoolExecutor

CACHE_SHORT_SIDE = 400
SMALL_DIR = "/content/skincon_small"


def _resize_one(args):
    src, dst = args
    if os.path.exists(dst):
        return
    with Image.open(src) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = CACHE_SHORT_SIDE / min(w, h)
        if s < 1:
            im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
        im.save(dst, quality=95)


os.makedirs(SMALL_DIR, exist_ok=True)
DATA["small_path"] = [os.path.join(SMALL_DIR, f"{i}.jpg") for i in DATA["image_id"]]
t0 = time.time()
jobs = list(zip(DATA["path"], DATA["small_path"]))
with ThreadPoolExecutor(IO_THREADS) as ex:
    for k, _ in enumerate(ex.map(_resize_one, jobs), 1):
        if k % 1000 == 0 or k == len(jobs):
            original_print(f"  resized {k}/{len(jobs)} images, {time.time() - t0:.0f} s", flush=True)

  resized 1000/3863 images, 11 s
  resized 2000/3863 images, 22 s
  resized 3000/3863 images, 31 s
  resized 3863/3863 images, 48 s


## Training library

In [5]:
# ============================================================
# TRAINING LIBRARY
# Concept predictor with a configurable backbone, regularization and schedule, mixed precision, checkpoints that
# resume inside a trial, concept metrics, per-concept thresholds and the five diagnosis heads of the analysis.
# ============================================================
import copy
import glob
import shutil
import dataclasses
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
TOTAL_EPOCHS = 40
RUNG_EPOCHS = 12
HEAD_SPECS = [("V12", "affine"), ("V12", "mlp"), ("V12", "deep"), ("V16", "mlp"), ("V20", "mlp")]
HEAD_HIDDEN = 128
os.makedirs(TUNING_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
LOG_FILE = os.path.join(TUNING_PATH, "trials.csv")


@dataclasses.dataclass
class TrialConfig:
    name: str
    tunable_pct: int = 75            # share of backbone blocks that are trained (from the top)
    batch_size: int = 16
    lr: float = 5e-4                 # learning rate of the concept heads
    backbone_lr_mult: float = 1.0    # backbone learning rate = lr * backbone_lr_mult
    optimizer: str = "adam"          # adam (published) or adamw
    weight_decay: float = 0.0
    head_dropout: float = 0.0        # dropout on the pooled features before the concept heads
    label_smoothing: float = 0.0
    stochastic_depth: float = 0.2    # torchvision default for efficientnet
    augment: str = "standard"        # standard (published) or strong (adds random erasing, stronger RandAugment)
    img_size: int = 224
    warmup_epochs: int = 0
    concept_weighting: bool = True   # class weights in the concept loss
    train_sources: tuple = ("fitzpatrick17k", "ddi")

    def key(self):
        return self.name


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True       # speed, runs stay reproducible up to GPU nondeterminism
    os.environ["PYTHONHASHSEED"] = str(seed)


BACKBONES = {"efficientnet_b0": ("efficientnet_b0", "EfficientNet_B0_Weights", 1280),
             "efficientnet_b3": ("efficientnet_b3", "EfficientNet_B3_Weights", 1536),
             "efficientnet_b4": ("efficientnet_b4", "EfficientNet_B4_Weights", 1792)}


class ConceptModel(nn.Module):
    def __init__(self, cfg, concepts, pretrained=True):
        super().__init__()
        fn, wname, dim = BACKBONES[BACKBONE_NAME]
        m = getattr(models, fn)(weights=getattr(models, wname).DEFAULT if pretrained else None,
                                stochastic_depth_prob=cfg.stochastic_depth)
        self.features = m.features
        total = len(self.features)
        freeze_up_to = int(round(total * (100 - cfg.tunable_pct) / 100))
        for i, layer in enumerate(self.features):
            for p in layer.parameters():
                p.requires_grad = i >= freeze_up_to
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(cfg.head_dropout)
        self.concepts = list(concepts)
        self.heads = nn.ModuleDict({c: nn.Linear(dim, 2) for c in self.concepts})

    def forward(self, x):
        z = self.drop(self.pool(self.features(x)).flatten(1))
        return [self.heads[c](z) for c in self.concepts]


class SkinConDataset(Dataset):
    def __init__(self, df, concepts, transform):
        self.paths = df["small_path"].tolist()
        self.c = torch.tensor(df[concepts].values, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        return self.transform(img), self.c[i]


NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])


def train_transforms(cfg):
    t = [transforms.RandomResizedCrop(cfg.img_size, scale=(0.7 if cfg.augment == "strong" else 0.8, 1.0)),
         transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(90),
         transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15)], p=0.5),
         transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
         transforms.RandAugment(num_ops=2, magnitude=12 if cfg.augment == "strong" else 9),
         transforms.ToTensor(), NORM]
    if cfg.augment == "strong":
        t.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    return transforms.Compose(t)


def eval_transforms(cfg):
    return transforms.Compose([transforms.Resize((cfg.img_size, cfg.img_size)), transforms.ToTensor(), NORM])


def concept_weights(df, concepts):
    out = []
    for c in concepts:
        pos = min(max(float(df[c].mean()), 1e-3), 1 - 1e-3)
        out.append(torch.tensor([0.5 / (1 - pos), 0.5 / pos], dtype=torch.float32, device=DEVICE))
    return out


def make_optimizer(cfg, model):
    back = [p for p in model.features.parameters() if p.requires_grad]
    head = list(model.heads.parameters())
    groups = [{"params": back, "lr": cfg.lr * cfg.backbone_lr_mult}, {"params": head, "lr": cfg.lr}]
    if cfg.optimizer == "adamw":
        return optim.AdamW(groups, weight_decay=cfg.weight_decay)
    return optim.Adam(groups, weight_decay=cfg.weight_decay)


def lr_factor(cfg, epoch):
    """Linear warmup, then cosine decay to 2% over TOTAL_EPOCHS (epoch counts from 0)."""
    if epoch < cfg.warmup_epochs:
        return (epoch + 1) / (cfg.warmup_epochs + 1)
    t = (epoch - cfg.warmup_epochs) / max(1, TOTAL_EPOCHS - cfg.warmup_epochs)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))


@torch.no_grad()
def concept_probs(model, df, cfg, batch=64):
    model.eval()
    dl = DataLoader(SkinConDataset(df, CONCEPTS, eval_transforms(cfg)), batch_size=batch, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    out = []
    for x, _ in dl:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            lg = model(x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last))
        out.append(torch.stack([torch.softmax(l.float(), 1)[:, 1] for l in lg], 1).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


def concept_metrics(prob, df, thresholds=None):
    t = df[CONCEPTS].values
    thr = np.full(len(CONCEPTS), 0.5) if thresholds is None else np.asarray(thresholds)
    hard = (prob >= thr[None, :]).astype(int)
    auc = [roc_auc_score(t[:, j], prob[:, j]) if 0 < t[:, j].sum() < len(t) else np.nan for j in range(len(CONCEPTS))]
    f1 = [f1_score(t[:, j], hard[:, j], average="macro", zero_division=0) for j in range(len(CONCEPTS))]
    ba = [balanced_accuracy_score(t[:, j], hard[:, j]) if 0 < t[:, j].sum() < len(t) else np.nan
          for j in range(len(CONCEPTS))]
    return {"concept_macro_auroc": float(np.nanmean(auc)), "concept_macro_f1": float(np.mean(f1)),
            "concept_balanced_accuracy": float(np.nanmean(ba)),
            "concept_macro_auroc_V12": float(np.nanmean([auc[CONCEPTS.index(c)] for c in VOCABS["V12"]]))}


def youden_thresholds(prob, df):
    """Per-concept threshold that maximizes balanced accuracy on the given (validation) images."""
    t = df[CONCEPTS].values
    thr = np.full(len(CONCEPTS), 0.5)
    grid = np.linspace(0.05, 0.95, 91)
    for j in range(len(CONCEPTS)):
        if 0 < t[:, j].sum() < len(t):
            ba = [balanced_accuracy_score(t[:, j], (prob[:, j] >= g).astype(int)) for g in grid]
            thr[j] = grid[int(np.argmax(ba))]
    return thr


# ---------------- diagnosis heads (identical architecture to the analysis notebook) ----------------

def one_hot_blocks(hard):
    L, n = hard.shape
    X = np.zeros((L, 2 * n))
    X[np.arange(L)[:, None], 2 * np.arange(n)[None, :] + hard] = 1.0
    return X


def make_head(kind, n, dropout=0.3):
    d, h = 2 * n, HEAD_HIDDEN
    if kind == "affine":
        return nn.Sequential(nn.Linear(d, 2))
    if kind == "mlp":
        return nn.Sequential(nn.Linear(d, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(dropout), nn.Linear(h, 2))
    layers, w = [], d
    for _ in range(3):
        layers += [nn.Linear(w, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(dropout)]
        w = h
    return nn.Sequential(*layers, nn.Linear(h, 2))


def head_spec(head):
    spec = []
    for m in head:
        if isinstance(m, nn.Linear):
            spec.append(("linear", m.weight.detach().double().cpu().numpy(), m.bias.detach().double().cpu().numpy()))
        elif isinstance(m, nn.LayerNorm):
            spec.append(("layernorm", m.weight.detach().double().cpu().numpy(), m.bias.detach().double().cpu().numpy(),
                         float(m.eps)))
        elif isinstance(m, nn.ReLU):
            spec.append(("relu",))
    return spec


def train_head(kind, Xtr_states, ytr, Xva_states, yva, seed, weight_decay=1e-4, dropout=0.3,
               epochs=300, patience=40, lr=1e-3, batch=64):
    set_seed(seed)
    n = Xtr_states.shape[1]
    Xtr = torch.tensor(one_hot_blocks(Xtr_states), dtype=torch.float32)
    Xva = torch.tensor(one_hot_blocks(Xva_states), dtype=torch.float32)
    yt, yv = torch.tensor(ytr, dtype=torch.long), torch.tensor(yva, dtype=torch.long)
    cw = torch.tensor(len(yt) / (2.0 * np.bincount(ytr, minlength=2).clip(1)), dtype=torch.float32)
    head = make_head(kind, n, dropout)
    opt = optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    g = torch.Generator()
    g.manual_seed(seed)
    best, state, best_ep, wait = np.inf, None, 0, 0
    for ep in range(1, epochs + 1):
        head.train()
        perm = torch.randperm(len(yt), generator=g)
        for i in range(0, len(yt), batch):
            b = perm[i:i + batch]
            opt.zero_grad()
            F.cross_entropy(head(Xtr[b]), yt[b], weight=cw).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            vl = F.cross_entropy(head(Xva), yv, weight=cw).item()
        if vl < best - 1e-6:
            best, state, best_ep, wait = vl, copy.deepcopy(head.state_dict()), ep, 0
        else:
            wait += 1
            if wait >= patience:
                break
    head.load_state_dict(state)
    head.eval()
    return head, best_ep


def head_logit(head, states):
    with torch.no_grad():
        o = head(torch.tensor(one_hot_blocks(states), dtype=torch.float32))
    return (o[:, 1] - o[:, 0]).double().numpy()


def label_metrics(y, s):
    pred = (s > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"), "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1), "auroc": roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan}


def primary_head_score(prob_tr, prob_va, df_tr, df_va, seed, thresholds=None, head_input="predicted"):
    """Validation metrics of the published two-layer head on V12, the ranking criterion of the search."""
    cols = [CONCEPTS.index(c) for c in VOCABS["V12"]]
    thr = np.full(len(CONCEPTS), 0.5) if thresholds is None else thresholds
    htr = (df_tr[CONCEPTS].values if head_input == "annotated" else (prob_tr >= thr).astype(int))[:, cols]
    hva = (prob_va >= thr).astype(int)[:, cols]
    head, ep = train_head("mlp", htr, df_tr["malignant"].values, hva, df_va["malignant"].values, seed)
    return label_metrics(df_va["malignant"].values, head_logit(head, hva))


# ---------------- label monitor for the progress output ----------------
_ANNOTATED_HEADS = {}


def annotated_label_head(tr, va, seed, sources):
    """Diagnosis head fitted once on the annotated V12 concepts, it gives a label score after every epoch at no extra cost."""
    key = (seed, tuple(sources))
    if key not in _ANNOTATED_HEADS:
        cols = [CONCEPTS.index(c) for c in VOCABS["V12"]]
        head, _ = train_head("mlp", tr[CONCEPTS].values[:, cols], tr["malignant"].values,
                             va[CONCEPTS].values[:, cols], va["malignant"].values, seed)
        _ANNOTATED_HEADS[key] = (head, cols)
    return _ANNOTATED_HEADS[key]


# ---------------- trials with checkpoints ----------------

def trial_dir(cfg, seed):
    d = os.path.join(TUNING_PATH, "trials", f"{cfg.name}_seed{seed}")
    os.makedirs(d, exist_ok=True)
    return d


def log_row(row):
    df = pd.DataFrame([row])
    df.to_csv(LOG_FILE, mode="a", header=not os.path.exists(LOG_FILE), index=False)


def run_trial(cfg, seed, until_epoch, splits):
    """
    Train cfg with seed up to until_epoch of its TOTAL_EPOCHS schedule, resuming from the last checkpoint.
    The weights with the best validation concept macro AUROC so far are kept in best.pth.
    Returns the trial state (history, best epoch, best validation concept AUROC).
    """
    d = trial_dir(cfg, seed)
    last, best_path, hist_path = os.path.join(d, "last.pth"), os.path.join(d, "best.pth"), os.path.join(d, "history.csv")
    with open(os.path.join(d, "config.json"), "w") as fh:
        json.dump(dataclasses.asdict(cfg), fh, indent=1)
    tr = splits["train"][splits["train"]["source"].isin(cfg.train_sources)].reset_index(drop=True)
    va = splits["valid"]
    ann_head, ann_cols = annotated_label_head(tr, va, seed, cfg.train_sources)
    set_seed(seed)
    model = ConceptModel(cfg, CONCEPTS).to(DEVICE).to(memory_format=torch.channels_last)
    opt = make_optimizer(cfg, model)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    start, best_auc, best_ep, history = 0, -1.0, -1, []
    if os.path.exists(last):
        ck = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best_auc, best_ep, history = ck["epoch"], ck["best_auc"], ck["best_ep"], ck["history"]
        torch.set_rng_state(ck["rng_cpu"].cpu())
        if torch.cuda.is_available() and ck.get("rng_cuda") is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in ck["rng_cuda"]])
    if start >= until_epoch:
        return {"history": history, "best_ep": best_ep, "best_auc": best_auc, "dir": d, "train_images": len(tr)}
    g = torch.Generator()
    g.manual_seed(seed * 1000 + start)
    dl = DataLoader(SkinConDataset(tr, CONCEPTS, train_transforms(cfg)), batch_size=cfg.batch_size, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, generator=g, persistent_workers=NUM_WORKERS > 0)
    wts = concept_weights(tr, CONCEPTS) if cfg.concept_weighting else [None] * len(CONCEPTS)
    base_lrs = [grp["lr"] for grp in make_optimizer(cfg, model).param_groups]
    for ep in range(start, until_epoch):
        t0 = time.time()
        for grp, b in zip(opt.param_groups, base_lrs):
            grp["lr"] = b * lr_factor(cfg, ep)
        model.train()
        tot, nb = 0.0, 0
        for x, c in dl:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            c = c.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                lg = model(x)
                loss = torch.stack([F.cross_entropy(lg[j].float(), c[:, j], weight=wts[j],
                                                    label_smoothing=cfg.label_smoothing)
                                    for j in range(len(CONCEPTS))]).mean()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item(); nb += 1
        pva = concept_probs(model, va, cfg)
        m = concept_metrics(pva, va)
        rec = {"epoch": ep + 1, "train_loss": tot / max(nb, 1), "seconds": time.time() - t0, **m}
        history.append(rec)
        if m["concept_macro_auroc"] > best_auc:
            best_auc, best_ep = m["concept_macro_auroc"], ep + 1
            torch.save(model.state_dict(), best_path)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": ep + 1, "best_auc": best_auc, "best_ep": best_ep, "history": history,
                    "rng_cpu": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                    "config": dataclasses.asdict(cfg)}, last)
        pd.DataFrame(history).to_csv(hist_path, index=False)
        lab = label_metrics(va["malignant"].values,
                            head_logit(ann_head, (pva >= 0.5).astype(int)[:, ann_cols]))
        original_print(f"    {cfg.name} seed {seed} epoch {ep + 1:>2}/{TOTAL_EPOCHS}  loss {rec['train_loss']:.4f}  "
                       f"concepts AUROC {m['concept_macro_auroc']:.4f} F1 {m['concept_macro_f1']:.4f} "
                       f"BA {m['concept_balanced_accuracy']:.4f}  "
                       f"labels F1 {lab['macro_f1']:.4f} BA {lab['balanced_accuracy']:.4f}  "
                       f"{rec['seconds']:.0f} s", flush=True)
    del model, opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"history": history, "best_ep": best_ep, "best_auc": best_auc, "dir": d, "train_images": len(tr)}


def load_best(cfg, seed):
    model = ConceptModel(cfg, CONCEPTS, pretrained=False).to(DEVICE).to(memory_format=torch.channels_last)
    model.load_state_dict(torch.load(os.path.join(trial_dir(cfg, seed), "best.pth"), map_location=DEVICE,
                                     weights_only=False))
    return model


def evaluate_trial(cfg, seed, splits, stage, extra=None):
    """Concept probabilities of the best weights plus the validation score of the primary head."""
    d = trial_dir(cfg, seed)
    cache = os.path.join(d, "probs.npz")
    if os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(os.path.join(d, "best.pth")):
        z = np.load(cache)
        probs = {s: z[s] for s in ("train", "valid", "test")}
    else:
        model = load_best(cfg, seed)
        probs = {s: concept_probs(model, splits[s], cfg) for s in ("train", "valid", "test")}
        np.savez_compressed(cache, **probs)
        del model
    tr_mask = splits["train"]["source"].isin(cfg.train_sources).values
    df_tr = splits["train"][tr_mask].reset_index(drop=True)
    head = primary_head_score(probs["train"][tr_mask], probs["valid"], df_tr, splits["valid"], seed)
    row = {"stage": stage, "config": cfg.name, "seed": seed, **(extra or {}),
           **{f"valid_{k}": v for k, v in concept_metrics(probs["valid"], splits["valid"]).items()},
           **{f"valid_head_{k}": v for k, v in head.items()}}
    return row, probs


class Budget:
    """GPU time spent so far, summed from the epoch durations recorded by every trial (survives restarts)."""

    def __init__(self, hours):
        self.limit = hours * 3600

    def used(self):
        tot = 0.0
        for f in glob.glob(os.path.join(TUNING_PATH, "trials", "*", "history.csv")):
            try:
                tot += pd.read_csv(f)["seconds"].sum()
            except Exception:
                pass
        return tot * 1.1                  # evaluation passes add roughly 10%

    def epoch_seconds(self, img_size):
        """Median epoch duration recorded so far at this input size (None before the first epoch)."""
        vals = []
        for f in glob.glob(os.path.join(TUNING_PATH, "trials", "*", "history.csv")):
            try:
                h = pd.read_csv(f)
                with open(os.path.join(os.path.dirname(f), "config.json")) as fh:
                    size = json.load(fh)["img_size"]
            except Exception:
                continue
            if size == img_size:
                vals += h["seconds"].tolist()
        return float(np.median(vals)) if vals else None

    def fits(self, epochs, img_size, reserve_hours=0.0):
        sec = self.epoch_seconds(img_size) or self.epoch_seconds(224)
        if sec is None:
            return True
        sec *= (img_size / 224) ** 2 if self.epoch_seconds(img_size) is None else 1.0
        return self.used() + 1.1 * epochs * sec + reserve_hours * 3600 <= self.limit

    def report(self, what=""):
        original_print(f"  [budget] {self.used() / 3600:.2f} h of {self.limit / 3600:.1f} h used {what}", flush=True)


BUDGET = Budget(TIME_BUDGET_HOURS)
SPLITS = {s: DATA[DATA["split"] == s].reset_index(drop=True) for s in ("train", "valid", "test")}
original_print(f"Device {DEVICE}, training images {len(SPLITS['train'])} "
               f"({int((SPLITS['train']['source'] == 'ddi').sum())} from DDI)")

Device cuda, training images 2316 (382 from DDI)


## Stage A. Search with successive halving (seed 42, training on Fitzpatrick17k plus DDI)

The base setting unfreezes the whole backbone, trains with batches of 8, AdamW with weight decay 0.05, dropout 0.5 before the concept heads, no label smoothing, no class weights in the concept loss, stochastic depth 0.4, stronger augmentation and 300 pixel inputs. Every other configuration changes one factor of the base setting, so the table also shows which factor matters. Configurations run in priority order, and the budget guard skips the last ones if time runs short.

In [6]:
# ============================================================
# STAGE A. SEARCH
# ============================================================
BASE = TrialConfig("base", tunable_pct=100, batch_size=8, lr=3e-4, backbone_lr_mult=1.0, optimizer="adamw",
                   weight_decay=0.05, head_dropout=0.5, label_smoothing=0.0, stochastic_depth=0.4, augment="strong",
                   img_size=300, warmup_epochs=1, concept_weighting=False)
R = dataclasses.replace
CONFIGS = [BASE,
           R(BASE, name="base_batch16", batch_size=16),
           R(BASE, name="base_lr1e-3", lr=1e-3),
           R(BASE, name="base_bb_lr0.3", backbone_lr_mult=0.3),
           R(BASE, name="base_moderate_reg", weight_decay=0.01, head_dropout=0.2, stochastic_depth=0.2)]
CFG_BY_NAME = {c.name: c for c in CONFIGS}
SEARCH_SEED = SEEDS[0]
PROMOTE = 3
SELECTION_FILE = os.path.join(TUNING_PATH, "selection.json")


def load_selection():
    if os.path.exists(SELECTION_FILE):
        with open(SELECTION_FILE) as fh:
            return json.load(fh)
    return {}


def save_selection(**kw):
    d = load_selection()
    d.update(kw)
    with open(SELECTION_FILE, "w") as fh:
        json.dump(d, fh, indent=1)


# ---- rung 1, every configuration for RUNG_EPOCHS epochs (reserve time for rung 2, Stage B and Stage C) ----
RESERVE_H = 4.0
rung1 = []
for cfg in CONFIGS:
    if not BUDGET.fits(RUNG_EPOCHS, cfg.img_size, reserve_hours=RESERVE_H) and cfg.name != "base":
        original_print(f"  skipping {cfg.name}, the projected time exceeds the budget")
        continue
    original_print(f"\nStage A, rung 1: {cfg.name}")
    run_trial(cfg, SEARCH_SEED, RUNG_EPOCHS, SPLITS)
    row, _ = evaluate_trial(cfg, SEARCH_SEED, SPLITS, "A_rung1", {"epochs": RUNG_EPOCHS})
    rung1.append(row)
    BUDGET.report(f"after {cfg.name}")
RUNG1 = pd.DataFrame(rung1).sort_values(["valid_head_auroc", "valid_concept_macro_auroc"], ascending=False)
RUNG1.to_csv(os.path.join(TUNING_PATH, "stageA_rung1.csv"), index=False)
original_print("\nStage A, rung 1 (sorted by the validation AUROC of the primary head)")
original_print(RUNG1[["config", "valid_head_auroc", "valid_head_balanced_accuracy", "valid_concept_macro_auroc",
                      "valid_concept_macro_f1"]].round(4).to_string(index=False))

# ---- rung 2, the best PROMOTE configurations continue to TOTAL_EPOCHS from their checkpoints ----
sel = load_selection()
promoted = sel.get("promoted") or RUNG1["config"].head(PROMOTE).tolist()
save_selection(promoted=promoted)
rung2 = []
for name in promoted:
    cfg = CFG_BY_NAME[name]
    original_print(f"\nStage A, rung 2: {name}")
    run_trial(cfg, SEARCH_SEED, TOTAL_EPOCHS, SPLITS)
    row, _ = evaluate_trial(cfg, SEARCH_SEED, SPLITS, "A_rung2", {"epochs": TOTAL_EPOCHS})
    rung2.append(row)
    BUDGET.report(f"after {name}")
RUNG2 = pd.DataFrame(rung2).sort_values(["valid_head_auroc", "valid_concept_macro_auroc"], ascending=False)
RUNG2.to_csv(os.path.join(TUNING_PATH, "stageA_rung2.csv"), index=False)
BEST_NAME = load_selection().get("best_config") or RUNG2.iloc[0]["config"]
save_selection(best_config=BEST_NAME)
original_print("\nStage A, rung 2")
original_print(RUNG2[["config", "valid_head_auroc", "valid_head_balanced_accuracy", "valid_concept_macro_auroc",
                      "valid_concept_macro_f1"]].round(4).to_string(index=False))
original_print(f"\nBest configuration: {BEST_NAME}")
# checkpoints of the configurations that were not promoted are no longer needed
for cfg in CONFIGS:
    if cfg.name not in promoted:
        f = os.path.join(trial_dir(cfg, SEARCH_SEED), "last.pth")
        if os.path.exists(f):
            os.remove(f)


Stage A, rung 1: base
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 206MB/s]


    base seed 42 epoch  1/40  loss 0.3265  concepts AUROC 0.6839 F1 0.5183 BA 0.5297  labels F1 0.4972 BA 0.4988  147 s
    base seed 42 epoch  2/40  loss 0.2848  concepts AUROC 0.7807 F1 0.5491 BA 0.5505  labels F1 0.5333 BA 0.5386  62 s
    base seed 42 epoch  3/40  loss 0.2650  concepts AUROC 0.8227 F1 0.6067 BA 0.6201  labels F1 0.5201 BA 0.5268  65 s
    base seed 42 epoch  4/40  loss 0.2569  concepts AUROC 0.8255 F1 0.6396 BA 0.6282  labels F1 0.5438 BA 0.5705  65 s
    base seed 42 epoch  5/40  loss 0.2500  concepts AUROC 0.8345 F1 0.6266 BA 0.6189  labels F1 0.5131 BA 0.5353  64 s
    base seed 42 epoch  6/40  loss 0.2445  concepts AUROC 0.8504 F1 0.6526 BA 0.6535  labels F1 0.5221 BA 0.5382  66 s
    base seed 42 epoch  7/40  loss 0.2381  concepts AUROC 0.8429 F1 0.6593 BA 0.6427  labels F1 0.5448 BA 0.5712  64 s
    base seed 42 epoch  8/40  loss 0.2323  concepts AUROC 0.8487 F1 0.6553 BA 0.6635  labels F1 0.5542 BA 0.5982  63 s
    base seed 42 epoch  9/40  loss 0.2281  conc

## Stage B. Does DDI help?

The best configuration is retrained on the Fitzpatrick17k training images alone, with the same seed. Both arms select their epoch on the same validation images and are compared on the same validation plus test images, overall and per source. The decision uses validation only: DDI stays in training unless the Fitzpatrick17k-only arm reaches a higher validation AUROC of the primary head. Test results are reported, never used.

In [7]:
# ============================================================
# STAGE B. DDI ABLATION
# ============================================================

def arm_metrics(cfg, seed, probs, arm):
    """Primary head of one arm, evaluated on validation plus test images, overall and per source."""
    tr_mask = SPLITS["train"]["source"].isin(cfg.train_sources).values
    df_tr = SPLITS["train"][tr_mask].reset_index(drop=True)
    cols = [CONCEPTS.index(c) for c in VOCABS["V12"]]
    htr = (probs["train"][tr_mask] >= 0.5).astype(int)[:, cols]
    hva = (probs["valid"] >= 0.5).astype(int)[:, cols]
    head, _ = train_head("mlp", htr, df_tr["malignant"].values, hva, SPLITS["valid"]["malignant"].values, seed)
    rows = []
    for split in ("valid", "test"):
        df = SPLITS[split]
        s = head_logit(head, (probs[split] >= 0.5).astype(int)[:, cols])
        for src in ("all", "fitzpatrick17k", "ddi"):
            m = np.ones(len(df), bool) if src == "all" else (df["source"] == src).values
            rows.append({"arm": arm, "split": split, "source": src, "images": int(m.sum()),
                         **label_metrics(df["malignant"].values[m], s[m]),
                         **{k: v for k, v in concept_metrics(probs[split][m], df[m]).items()}})
    return rows


BEST = CFG_BY_NAME[BEST_NAME]
FITZ_ONLY = dataclasses.replace(BEST, name=f"{BEST_NAME}_fitz_only", train_sources=("fitzpatrick17k",))
original_print(f"\nStage B: {FITZ_ONLY.name}")
run_trial(BEST, SEARCH_SEED, TOTAL_EPOCHS, SPLITS)
run_trial(FITZ_ONLY, SEARCH_SEED, TOTAL_EPOCHS, SPLITS)
_, probs_ddi = evaluate_trial(BEST, SEARCH_SEED, SPLITS, "B")
_, probs_fitz = evaluate_trial(FITZ_ONLY, SEARCH_SEED, SPLITS, "B")
ABLATION = pd.DataFrame(arm_metrics(BEST, SEARCH_SEED, probs_ddi, "Fitzpatrick17k + DDI") +
                        arm_metrics(FITZ_ONLY, SEARCH_SEED, probs_fitz, "Fitzpatrick17k only"))
ABLATION.to_csv(os.path.join(TUNING_PATH, "stageB_ddi_ablation.csv"), index=False)
original_print(ABLATION[["arm", "split", "source", "images", "auroc", "balanced_accuracy", "sensitivity", "specificity",
                         "concept_macro_auroc", "concept_macro_f1"]].round(4).to_string(index=False))
va = ABLATION[(ABLATION.split == "valid") & (ABLATION.source == "all")].set_index("arm")["auroc"]
USE_DDI_TRAIN = load_selection().get("use_ddi_in_training")
if USE_DDI_TRAIN is None:
    USE_DDI_TRAIN = bool(va["Fitzpatrick17k + DDI"] >= va["Fitzpatrick17k only"])
    save_selection(use_ddi_in_training=USE_DDI_TRAIN)
FINAL = BEST if USE_DDI_TRAIN else FITZ_ONLY
original_print(f"\nDDI in training: {USE_DDI_TRAIN}, final configuration {FINAL.name}")
BUDGET.report("after Stage B")


Stage B: base_bb_lr0.3_fitz_only
    base_bb_lr0.3_fitz_only seed 42 epoch  1/40  loss 0.3504  concepts AUROC 0.5932 F1 0.4793 BA 0.5062  labels F1 0.4570 BA 0.4865  52 s
    base_bb_lr0.3_fitz_only seed 42 epoch  2/40  loss 0.2969  concepts AUROC 0.7014 F1 0.5378 BA 0.5420  labels F1 0.4491 BA 0.4591  51 s
    base_bb_lr0.3_fitz_only seed 42 epoch  3/40  loss 0.2805  concepts AUROC 0.7613 F1 0.6021 BA 0.6009  labels F1 0.5198 BA 0.5546  57 s
    base_bb_lr0.3_fitz_only seed 42 epoch  4/40  loss 0.2700  concepts AUROC 0.7950 F1 0.6091 BA 0.6116  labels F1 0.4831 BA 0.5098  56 s
    base_bb_lr0.3_fitz_only seed 42 epoch  5/40  loss 0.2593  concepts AUROC 0.8087 F1 0.6247 BA 0.6294  labels F1 0.5167 BA 0.5535  56 s
    base_bb_lr0.3_fitz_only seed 42 epoch  6/40  loss 0.2580  concepts AUROC 0.8204 F1 0.6289 BA 0.6235  labels F1 0.5405 BA 0.5785  56 s
    base_bb_lr0.3_fitz_only seed 42 epoch  7/40  loss 0.2476  concepts AUROC 0.8288 F1 0.6507 BA 0.6645  labels F1 0.5622 BA 0.6098  56 s


## Stage C. Head options and the final models

Cheap choices are made on validation images with the seed-42 model of the final configuration. The vocabulary size (12, 16 or 20 concepts) is chosen here as well, because the extractor always predicts 20 concepts and only the label classifier changes. Heads either learn from the predicted states of the training images (sequential) or from their annotated concepts (independent). Hard states either use the argmax of each concept head or a per-concept threshold that maximizes validation balanced accuracy. The label classifier is trained after the concept extractor, so it has its own weight decay, learning rate and batch size, all chosen here. Then three seeds of the final configuration are trained, and each gets the five heads of the analysis. The exports go to `models` in the format that the analysis notebook reads. Each seed also gets one file, `model_<backbone>_seed<seed>.pth`, that holds everything needed to predict with it.

In [8]:
# ============================================================
# STAGE C. HEAD OPTIONS, FINAL SEEDS, EXPORT
# ============================================================

def head_inputs(probs, thr, head_input, tr_mask):
    df_tr = SPLITS["train"][tr_mask].reset_index(drop=True)
    htr = df_tr[CONCEPTS].values if head_input == "annotated" else (probs["train"][tr_mask] >= thr).astype(int)
    return df_tr, htr, {s: (probs[s] >= thr).astype(int) for s in ("valid", "test")}


# ---- head options on validation ----
tr_mask = SPLITS["train"]["source"].isin(FINAL.train_sources).values
_, probs42 = evaluate_trial(FINAL, SEARCH_SEED, SPLITS, "C")
thr_youden = youden_thresholds(probs42["valid"], SPLITS["valid"])
rows = []
for head_input in ("predicted", "annotated"):
    for thr_name, thr in (("argmax", np.full(len(CONCEPTS), 0.5)), ("validation_threshold", thr_youden)):
        for wd, hlr, hb, vn in [(w, l, b, v) for w in (1e-4, 1e-2) for l in (3e-4, 1e-3, 3e-3) for b in (16, 32)
                                for v in VOCABS]:
            cols_v = [CONCEPTS.index(c) for c in VOCABS[vn]]
            df_tr, htr, hev = head_inputs(probs42, thr, head_input, tr_mask)
            head, ep = train_head("mlp", htr[:, cols_v], df_tr["malignant"].values, hev["valid"][:, cols_v],
                                  SPLITS["valid"]["malignant"].values, SEARCH_SEED, weight_decay=wd, lr=hlr, batch=hb)
            m = label_metrics(SPLITS["valid"]["malignant"].values, head_logit(head, hev["valid"][:, cols_v]))
            rows.append({"vocab": vn, "head_input": head_input, "states": thr_name, "weight_decay": wd, "head_lr": hlr, "head_batch": hb, "best_epoch": ep,
                         **{f"valid_{k}": v for k, v in m.items()}})
HEAD_GRID = pd.DataFrame(rows).sort_values("valid_auroc", ascending=False)
HEAD_GRID.to_csv(os.path.join(TUNING_PATH, "stageC_head_options.csv"), index=False)
original_print(HEAD_GRID.round(4).to_string(index=False))
sel = load_selection()
if "head_options" not in sel:
    top = HEAD_GRID.iloc[0]
    save_selection(head_options={"vocab": str(top["vocab"]), "head_input": top["head_input"], "states": top["states"],
                                 "weight_decay": float(top["weight_decay"]), "head_lr": float(top["head_lr"]),
                                 "head_batch": int(top["head_batch"])})
HOPT = load_selection()["head_options"]
original_print(f"\nHead options: {HOPT}")
BEST_HEAD = f"{HOPT.get('vocab', 'V12')}_mlp"      # the head of the selected vocabulary size, the model to use

# ---- final seeds ----
MANIFEST = {"backbone": BACKBONE_NAME, "split_file": SPLIT_FILE, "vocab_file": VOCAB_FILE, "concepts": CONCEPTS,
            "vocabularies": VOCABS, "concept_names": CONCEPT_NAMES, "head_specs": HEAD_SPECS,
            "training": {"HEAD_HIDDEN": HEAD_HIDDEN, "TOTAL_EPOCHS": TOTAL_EPOCHS,
                         "final_config": dataclasses.asdict(FINAL), "head_options": HOPT,
                         "selection": load_selection()},
            "seeds": {}}
metric_rows = []
for seed in SEEDS:
    t0 = time.time()
    original_print(f"\nFinal model, seed {seed}")
    run_trial(FINAL, seed, TOTAL_EPOCHS, SPLITS)
    _, probs = evaluate_trial(FINAL, seed, SPLITS, "C_final")
    thr = youden_thresholds(probs["valid"], SPLITS["valid"]) if HOPT["states"] == "validation_threshold" \
        else np.full(len(CONCEPTS), 0.5)
    hard = {s: (probs[s] >= thr).astype(int) for s in ("train", "valid", "test")}
    for s in ("train", "valid", "test"):
        metric_rows.append({"seed": seed, "model": "concepts", "split": s,
                            **concept_metrics(probs[s], SPLITS[s], thr), "best_epoch": None})
    df_tr = SPLITS["train"][tr_mask].reset_index(drop=True)
    htr_all = df_tr[CONCEPTS].values if HOPT["head_input"] == "annotated" else hard["train"][tr_mask]
    y = {s: SPLITS[s]["malignant"].values for s in ("train", "valid", "test")}
    heads, preds = {}, []
    for vname, kind in HEAD_SPECS:
        cols = [CONCEPTS.index(c) for c in VOCABS[vname]]
        head, ep = train_head(kind, htr_all[:, cols], df_tr["malignant"].values, hard["valid"][:, cols], y["valid"],
                              seed, weight_decay=HOPT["weight_decay"], lr=HOPT.get("head_lr", 1e-3), batch=HOPT.get("head_batch", 64))
        key = f"{vname}_{kind}"
        heads[key] = {"spec": head_spec(head), "state_dict": head.state_dict(), "kind": kind, "vocab": vname,
                      "columns": cols, "best_epoch": ep}
        for s in ("train", "valid", "test"):
            sc = head_logit(head, hard[s][:, cols])
            metric_rows.append({"seed": seed, "model": key, "split": s, **label_metrics(y[s], sc), "best_epoch": ep})
            preds.append(pd.DataFrame({"seed": seed, "head": key, "split": s, "image_id": SPLITS[s]["image_id"],
                                       "y_true": y[s], "logit": sc, "y_pred": (sc > 0).astype(int)}))
    feat_path = os.path.join(MODELS_PATH, f"states_{BACKBONE_NAME}_seed{seed}.npz")
    np.savez_compressed(feat_path, **{f"prob_{s}": probs[s] for s in probs}, **{f"hard_{s}": hard[s] for s in hard},
                        **{f"y_{s}": y[s] for s in y},
                        **{f"id_{s}": SPLITS[s]["image_id"].values.astype(str) for s in y},
                        **{f"group_{s}": SPLITS[s]["group"].values for s in y}, thresholds=thr)
    head_path = os.path.join(MODELS_PATH, f"heads_{BACKBONE_NAME}_seed{seed}.pth")
    torch.save(heads, head_path)
    pred_path = os.path.join(MODELS_PATH, f"predictions_{BACKBONE_NAME}_seed{seed}.csv")
    pd.concat(preds, ignore_index=True).to_csv(pred_path, index=False)
    ck_path = os.path.join(MODELS_PATH, f"concepts_{BACKBONE_NAME}_seed{seed}.pth")
    shutil.copyfile(os.path.join(trial_dir(FINAL, seed), "best.pth"), ck_path)
    bundle_path = os.path.join(MODELS_PATH, f"model_{BACKBONE_NAME}_seed{seed}.pth")
    torch.save({"backbone": BACKBONE_NAME, "seed": seed, "config": dataclasses.asdict(FINAL), "concepts": CONCEPTS,
                "vocabularies": VOCABS, "concept_names": CONCEPT_NAMES, "thresholds": thr,
                "head_input": HOPT["head_input"], "states": HOPT["states"], "best_head": BEST_HEAD, "heads": heads,
                "concept_model": torch.load(ck_path, map_location="cpu", weights_only=False)}, bundle_path)
    MANIFEST["seeds"][str(seed)] = {"concept_checkpoint": ck_path, "states": feat_path, "heads": head_path,
                                    "predictions": pred_path, "model": bundle_path, "seconds": time.time() - t0}
    with open(os.path.join(MODELS_PATH, "manifest_skincon.json"), "w") as fh:
        json.dump(MANIFEST, fh, indent=1, default=str)
    BUDGET.report(f"after seed {seed}")
METRICS = pd.DataFrame(metric_rows)
METRICS.to_csv(os.path.join(MODELS_PATH, "training_metrics.csv"), index=False)

vocab head_input               states  weight_decay  head_lr  head_batch  best_epoch  valid_accuracy  valid_balanced_accuracy  valid_macro_f1  valid_sensitivity  valid_specificity  valid_auroc
  V20  predicted validation_threshold        0.0001   0.0030          32           3          0.7283                   0.7467          0.6588             0.7754             0.7181       0.8100
  V20  predicted validation_threshold        0.0100   0.0030          32           3          0.7257                   0.7452          0.6565             0.7754             0.7150       0.8091
  V20  predicted validation_threshold        0.0100   0.0003          32          28          0.7646                   0.7546          0.6858             0.7391             0.7701       0.8091
  V20  predicted validation_threshold        0.0100   0.0003          16          12          0.7607                   0.7551          0.6833             0.7464             0.7638       0.8076
  V20  predicted validation_thresho

## Summary

Test metrics of the tuned models (mean and standard deviation over seeds), followed by the archive to send back.

In [9]:
# ============================================================
# SUMMARY AND ARCHIVE
# ============================================================
cols_head = ["auroc", "balanced_accuracy", "macro_f1", "sensitivity", "specificity"]


def summarize(df, label):
    heads = df[(df.model != "concepts") & (df.split == "test")]
    out = heads.groupby("model")[cols_head].agg(["mean", "std"])
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out = out.reset_index().assign(models=label)
    c = df[(df.model == "concepts") & (df.split == "test")]
    conc = {"models": label, "model": "concepts", "concept_macro_f1_mean": c["concept_macro_f1"].mean(),
            "concept_macro_f1_std": c["concept_macro_f1"].std()}
    if "concept_macro_auroc" in c:
        conc.update({"concept_macro_auroc_mean": c["concept_macro_auroc"].mean(),
                     "concept_macro_auroc_std": c["concept_macro_auroc"].std()})
    return pd.concat([out, pd.DataFrame([conc])], ignore_index=True)


frames = [summarize(METRICS, "tuned")]
COMPARISON = pd.concat(frames, ignore_index=True)
COMPARISON.to_csv(os.path.join(TUNING_PATH, "comparison_test.csv"), index=False)
show_cols = ["models", "model", "auroc_mean", "auroc_std", "balanced_accuracy_mean", "balanced_accuracy_std",
             "concept_macro_f1_mean", "concept_macro_auroc_mean"]
original_print("\nTest metrics of the tuned models")
original_print(COMPARISON[[c for c in show_cols if c in COMPARISON]].round(4).to_string(index=False))

# archive with every table, the trial histories and the selection
arch_dir = os.path.join(TUNING_PATH, "archive")
shutil.rmtree(arch_dir, ignore_errors=True)
os.makedirs(os.path.join(arch_dir, "histories"), exist_ok=True)
for f in glob.glob(os.path.join(TUNING_PATH, "*.csv")) + [SELECTION_FILE]:
    shutil.copy(f, arch_dir)
shutil.copy(os.path.join(MODELS_PATH, "training_metrics.csv"), os.path.join(arch_dir, "training_metrics_tuned.csv"))
shutil.copy(os.path.join(MODELS_PATH, "manifest_skincon.json"), arch_dir)
for d in glob.glob(os.path.join(TUNING_PATH, "trials", "*")):
    for f in ("history.csv", "config.json"):
        if os.path.exists(os.path.join(d, f)):
            shutil.copy(os.path.join(d, f), os.path.join(arch_dir, "histories", f"{os.path.basename(d)}_{f}"))
archive = shutil.make_archive(os.path.join(PROJECT_DIR, "tuning_outputs"), "zip", arch_dir)
original_print(f"\nArchive to send back: {archive}")
BUDGET.report("in total")

# ---- optimal hyperparameters of each model
original_print("\nOptimal hyperparameters")
original_print(f"  DDI images in training {USE_DDI_TRAIN}, final configuration {FINAL.name}")
for seed in SEEDS:
    h = pd.read_csv(os.path.join(trial_dir(FINAL, seed), "history.csv"))
    best_ep = int(h.loc[h["concept_macro_auroc"].idxmax(), "epoch"])
    original_print(f"\n  Model seed {seed}")
    original_print(f"    concept extractor, best epoch {best_ep} of {TOTAL_EPOCHS}")
    for k, v in dataclasses.asdict(FINAL).items():
        original_print(f"      {k:<20} {v}")
    original_print("    label classifier")
    for k, v in {"vocabulary size": f"{HOPT.get('vocab', 'V12')} ({len(VOCABS[HOPT.get('vocab', 'V12')])} concepts)",
                 "training input": HOPT["head_input"], "hard states": HOPT["states"],
                 "weight decay": HOPT["weight_decay"], "learning rate": HOPT.get("head_lr", 1e-3),
                 "batch size": HOPT.get("head_batch", 64), "hidden width": HEAD_HIDDEN}.items():
        original_print(f"      {k:<20} {v}")
    hb = METRICS[(METRICS.seed == seed) & (METRICS.split == "valid") & (METRICS.model != "concepts")]
    original_print("    best epoch of each label head  " + ", ".join(f"{m} {int(e)}" for m, e in zip(hb.model, hb.best_epoch)))



Test metrics of the tuned models
models      model  auroc_mean  auroc_std  balanced_accuracy_mean  balanced_accuracy_std  concept_macro_f1_mean  concept_macro_auroc_mean
 tuned V12_affine      0.7066     0.0154                  0.6523                 0.0136                    NaN                       NaN
 tuned   V12_deep      0.7279     0.0121                  0.6677                 0.0232                    NaN                       NaN
 tuned    V12_mlp      0.7309     0.0178                  0.6682                 0.0083                    NaN                       NaN
 tuned    V16_mlp      0.7380     0.0093                  0.6690                 0.0035                    NaN                       NaN
 tuned    V20_mlp      0.7738     0.0088                  0.7047                 0.0248                    NaN                       NaN
 tuned   concepts         NaN        NaN                     NaN                    NaN                 0.6378                    0.8493

Archiv

## Using a saved model

Each `model_<backbone>_seed<seed>.pth` file holds the concept extractor, the per-concept thresholds and the five label heads of one seed. The head of the selected vocabulary size is the default. The function below returns the concept probabilities, the concept states and the malignancy probability for a list of image files. Run the training library cell first, since it defines the model classes.


In [10]:
# ============================================================
# USING A SAVED MODEL
# ============================================================

@torch.no_grad()
def predict_images(bundle_path, image_paths, head=None):
    """Concept probabilities, concept states and malignancy probability of one seed for a list of image files."""
    b = torch.load(bundle_path, map_location="cpu", weights_only=False)
    cfg = TrialConfig(**{**b["config"], "train_sources": tuple(b["config"]["train_sources"])})
    model = ConceptModel(cfg, b["concepts"], pretrained=False)
    model.load_state_dict(b["concept_model"])
    model.to(DEVICE).eval()
    tf = eval_transforms(cfg)
    x = torch.stack([tf(Image.open(p).convert("RGB")) for p in image_paths]).to(DEVICE)
    prob = torch.stack([torch.softmax(l.float(), 1)[:, 1] for l in model(x)], 1).cpu().numpy()
    hard = (prob >= np.asarray(b["thresholds"])[None, :]).astype(int)
    h = b["heads"][head or b["best_head"]]
    net = make_head(h["kind"], len(h["columns"]))
    net.load_state_dict(h["state_dict"])
    net.eval()
    logit = head_logit(net, hard[:, h["columns"]])
    return {"concepts": b["concepts"], "concept_probs": prob, "concept_states": hard,
            "malignant_probability": 1 / (1 + np.exp(-logit))}


# example
# out = predict_images(f"{MODELS_PATH}/model_{BACKBONE_NAME}_seed42.pth", ["/content/example.jpg"])
# out["malignant_probability"]
